# Clase 02: Pipeline de entrenamiento en PyTorch

Esta notebook baja la teoria a codigo de trabajo real. La pregunta central no es solo como definir una red, sino como construir un pipeline que una datos, batches, modelo, perdida, optimizador y evaluacion sin dejar nada implicito.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset


from tools.notebook_utils import choose_value, configure_runtime, count_parameters, describe_tensor, plot_history

runtime = configure_runtime(seed=11)
print(runtime.summary())
        


## Por que PyTorch importa

La razon de ser de PyTorch en una clase inicial no es memorizar sintaxis. Es aprender a separar responsabilidades:

- los datos viven en `Dataset` y `DataLoader`,
- el modelo vive en `nn.Module`,
- la logica de entrenamiento vive en un loop explicito,
- la evaluacion debe usar `torch.inference_mode()` para evitar trabajo innecesario.

Ese desacople es lo que vuelve reproducible y mantenible un experimento.


In [ ]:
wine = load_wine()
X_train, X_temp, y_train, y_temp = train_test_split(
    wine.data,
    wine.target,
    test_size=0.3,
    random_state=11,
    stratify=wine.target,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=11,
    stratify=y_temp,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(describe_tensor('X_train', torch.tensor(X_train, dtype=torch.float32)))
print({'train_examples': len(X_train), 'val_examples': len(X_val), 'test_examples': len(X_test)})
        


In [ ]:
class WineDataset(Dataset):
    def __init__(self, features: np.ndarray, targets: np.ndarray) -> None:
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.features[index], self.targets[index]


train_loader = DataLoader(
    WineDataset(X_train, y_train),
    batch_size=int(choose_value(32, 16)),
    shuffle=True,
)
val_loader = DataLoader(WineDataset(X_val, y_val), batch_size=64)
test_loader = DataLoader(WineDataset(X_test, y_test), batch_size=64)


class WineMLP(torch.nn.Module):
    def __init__(self, input_dim: int, num_classes: int) -> None:
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, num_classes),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features)


model = WineMLP(input_dim=X_train.shape[1], num_classes=len(wine.target_names)).to(runtime.device)
print({'trainable_parameters': count_parameters(model)})
        


## Entrenar bien es hacer explicito lo invisible

En clase conviene ver el loop completo porque ahi aparecen decisiones de calidad:

- donde se hace `zero_grad`,
- donde se mueve el batch al dispositivo,
- donde se apaga el seguimiento de gradientes,
- que metricas importan de verdad para leer progreso.

`torch.compile` existe y puede ser util en ciertos contextos, pero no es requisito pedagogico para una primera implementacion estable.


In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-3)
epochs = int(choose_value(80, 25))
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for features, targets in train_loader:
        features = features.to(runtime.device)
        targets = targets.to(runtime.device)

        optimizer.zero_grad()
        logits = model(features)
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(features)

    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    val_running_loss = 0.0
    val_predictions = []
    val_targets = []
    with torch.inference_mode():
        for features, targets in val_loader:
            features = features.to(runtime.device)
            targets = targets.to(runtime.device)
            logits = model(features)
            val_running_loss += loss_fn(logits, targets).item() * len(features)
            val_predictions.extend(logits.argmax(dim=1).cpu().tolist())
            val_targets.extend(targets.cpu().tolist())

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_running_loss / len(val_loader.dataset))
    history['val_acc'].append(accuracy_score(val_targets, val_predictions))

plot_history(history, metrics=('train_loss', 'val_loss'), title='Pipeline en PyTorch: loss')
plt.figure(figsize=(8, 3))
plt.plot(history['val_acc'], label='val_acc')
plt.title('Accuracy de validacion')
plt.xlabel('Epoca')
plt.legend()
plt.tight_layout()
        


In [ ]:
model.eval()
test_predictions = []
test_targets = []
with torch.inference_mode():
    for features, targets in test_loader:
        features = features.to(runtime.device)
        logits = model(features)
        test_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        test_targets.extend(targets.tolist())

test_acc = accuracy_score(test_targets, test_predictions)
print({'test_acc': round(test_acc, 3)})

sample_batch = next(iter(test_loader))[0][:5].to(runtime.device)
with torch.inference_mode():
    sample_pred = model(sample_batch).argmax(dim=1).cpu().tolist()
print({'predicciones_ejemplo': sample_pred, 'clases': list(wine.target_names)})
        


## Para cerrar

### Lo importante de esta clase

- `Dataset` y `DataLoader` no son decorativos: ordenan el flujo de datos.
- `nn.Module` hace mantenible la arquitectura.
- `torch.inference_mode()` es la forma moderna y barata de evaluar.
- Si una metrica mejora pero la validacion empeora, el problema no es de sintaxis: es de generalizacion.

### Pruebas utiles

- cambia el tamano del batch y mira la estabilidad,
- reduce la red y compara costo versus accuracy,
- intenta entrenar sin normalizar y observa el impacto.
